# Практическое занятие 2. Извлечение данных из заявок на аренду жилья

Решение: LangChain + GigaChat извлекают количество проживающих из
неструктурированных текстов заявок.

Ключ доступа хранится в файле `.env` в переменной `GIGA_KEY` и не попадает
в репозиторий.

## 1. Подключение к GigaChat

In [1]:
import os

from dotenv import load_dotenv
from langchain_gigachat.chat_models import GigaChat

load_dotenv()
giga_key = os.getenv("GIGA_KEY")

if not giga_key:
    raise ValueError("Ключ GIGA_KEY не найден в .env")

llm = GigaChat(
    credentials=giga_key,
    model="GigaChat-2",
    verify_ssl_certs=False,
    temperature=0.2,
    max_tokens=1000,
)

print(llm.invoke("Привет! Как дела?").content)

Привет! Всё отлично, спасибо. Готов обсудить что-нибудь интересное или помочь разобраться с задачей. Чем займёмся?


## 2. Базовый промпт и цепочка LangChain

Промпт формулирует задачу, модель её выполняет, парсер `StrOutputParser`
возвращает результат как строку.

In [2]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

basic_prompt = PromptTemplate(
    input_variables=["text"],
    template="""
Проанализируй следующий текст заявки на аренду жилья и извлеки количество человек, которые будут проживать.
Текст заявки: {text}
Верни только число (целое число), соответствующее количеству проживающих.
Если количество не указано явно, постарайся определить его по контексту.
Количество человек:""",
)

chain = basic_prompt | llm | StrOutputParser()

test_texts = [
    "Ищу квартиру для семьи из четырех человек",
    "Нужна студия для одного",
    "Семья с двумя детьми ищет жилье",
]

for text in test_texts:
    print(f"Текст: {text}")
    print(f"Результат: {chain.invoke({'text': text})}")
    print("---")

Текст: Ищу квартиру для семьи из четырех человек


Результат: 4
---
Текст: Нужна студия для одного


Результат: 1
---
Текст: Семья с двумя детьми ищет жилье
Результат: 4
---


## 3. Обработка моих 15 заявок (вариант 17)

Заявки берутся из файла `rental_17.csv`. Ответы модели собираются в столбец
`result`, итоговая таблица сохраняется в `rental_with_results.csv`.

In [3]:
import pandas as pd

df = pd.read_csv("rental_17.csv", sep=";")

results = []
for _, row in df.iterrows():
    text = row["text"]
    try:
        results.append(chain.invoke({"text": text}))
    except Exception as error:
        results.append(f"ERROR: {error}")

df["result"] = results
df.to_csv("rental_with_results.csv", index=False, encoding="utf-8-sig")
df[["amount", "result"]]

,amount,result
0,6,6
1,2,2
2,4,4
3,3,3
4,3,3
5,3,3
6,3,3
7,4,4
8,4,4
9,2,2


## 4. Оценка точности базовой цепочки

Сравниваем ответы модели из столбца `result` с эталонным `amount`.

In [4]:
import pandas as pd

# Типы данных: столбец result хранит строки (object)
print(df.dtypes)

# Преобразуем ответы модели в числа; нечисловые ответы станут NaN
df["result_num"] = pd.to_numeric(df["result"], errors="coerce")
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")

correct = (df["result_num"] == df["amount"]).sum()
total = len(df)
errors = total - correct

print("Ошибок:", errors)
print("Точность: {:.1%}".format(correct / total))

# Строки, где ответ модели не совпал с эталоном
df.loc[df["result_num"] != df["amount"], ["amount", "result", "result_num"]]

amount    int64
text        str
result      str
dtype: object
Ошибок: 1
Точность: 93.3%


,amount,result,result_num
13,1,Из представленного текста невозможно однозначн...,NaN


## 5. Сравнение техник промптирования

Проверяем на тех же 15 заявках три техники: детализированные инструкции,
few-shot и Chain-of-Thought.

In [5]:
import re

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

detailed_prompt = PromptTemplate(
    input_variables=["text"],
    template="""
Ты — эксперт по анализу текстов заявок на аренду жилья. Твоя задача — извлечь точное количество людей, которые будут проживать в арендуемом жилье.
ИНСТРУКЦИИ:
1. Внимательно прочитай текст заявки
2. Найди все упоминания количества людей (прямые и косвенные)
3. Учти следующие правила:
- "семья с одним ребенком" = 3 человека (родители + ребенок)
- "семья с двумя детьми" = 4 человека (родители + дети)
- "пара", "двое", "молодая семья" = 2 человека
- "один", "студент", "индивидуально" = 1 человек
4. Если количество указано числом (например, "троих", "четверых"), используй это число
5. Верни ТОЛЬКО целое число без дополнительных символов или пояснений

Текст заявки: {text}
Количество проживающих:""",
)

few_shot_prompt = PromptTemplate(
    input_variables=["text"],
    template="""
Извлеки количество людей из текста заявки на аренду жилья. Изучи примеры:
ПРИМЕРЫ:
Заявка: "Семья из трех человек ищет двухкомнатную квартиру"
Количество: 3
Заявка: "Молодая пара снимет студию в центре города"
Количество: 2
Заявка: "Ищу однокомнатную квартиру для себя"
Количество: 1
Заявка: "Семья с одним ребенком рассматривает варианты жилья"
Количество: 3
Заявка: "Четверо студентов ищут четырехкомнатную квартиру"
Количество: 4

Теперь проанализируй эту заявку:
Заявка: {text}
Количество:""",
)

cot_prompt = PromptTemplate(
    input_variables=["text"],
    template="""
Проанализируй заявку на аренду жилья и определи количество проживающих, следуя пошаговому алгоритму:
Заявка: {text}
ШАГ 1: Найди все упоминания людей в тексте
ШАГ 2: Определи значение каждого упоминания
- "семья с N детьми" = 2 родителя + N детей
- "пара" = 2 человека
- числительные переведи в цифры
ШАГ 3: Сложи общее количество
ШАГ 4: Верни итоговое число

Пошаговый анализ:
ШАГ 1:
ШАГ 2:
ШАГ 3:
ШАГ 4: Итоговое число:""",
)

techniques = {
    "Детализированный": detailed_prompt,
    "Few-shot": few_shot_prompt,
    "Chain-of-Thought": cot_prompt,
}


def to_int(value):
    if not isinstance(value, str):
        return None
    numbers = re.findall(r"-?\d+", value)
    return int(numbers[-1]) if numbers else None


for name, prompt in techniques.items():
    tech_chain = prompt | llm | StrOutputParser()
    predictions = []
    for _, row in df.iterrows():
        try:
            predictions.append(to_int(tech_chain.invoke({"text": row["text"]})))
        except Exception as error:
            print("ERROR:", error)
            predictions.append(None)
    correct_tech = sum(1 for value, amount in zip(predictions, df["amount"]) if value == amount)
    print(f"{name}: {correct_tech}/{len(df)} = {correct_tech / len(df):.1%}")

Детализированный: 15/15 = 100.0%


Few-shot: 14/15 = 93.3%


Chain-of-Thought: 14/15 = 93.3%
